# Raw vs. Per-Layer-Normalized Head Selection: Does It Change the Causal Effect?

Every notebook in this project selects the top-K steered heads by **raw**
attention score (`rank_heads_by_score` sorts `(layer, head)` pairs by the raw,
un-normalized discovery score). The project also reports a **global max-normalized**
score (`score / score.max()`) for comparing magnitudes across checkpoints -- but
that's a single positive-constant rescaling, a strictly monotonic transform, so it
provably **cannot** change which heads land in the top-K. Raw-based and
global-max-normalized-based selection always pick the identical heads.

What genuinely *can* reorder heads is **per-layer normalization**
(`score / score.max(axis=head)`, dividing each layer's scores by that layer's own
max) -- this favors a head that stands out *within* its own layer even if its
absolute magnitude is smaller than a head in a naturally more attention-heavy
layer elsewhere in the network. This notebook tests whether that different
selection criterion actually changes the causal steering effect, across multiple
checkpoints from the Qwen and Gemma families already used elsewhere in this
project (`multistage_vis_head_causality.ipynb`).

**Models tested**: the checkpoints from those two families that showed a real,
non-degenerate causal signal in the multistage notebook (base/pt checkpoints are
excluded -- they score ~0 baseline accuracy regardless of head selection, so
there's no signal to differentiate a selection method against):

- `Qwen/Qwen3-VL-8B-Instruct`
- `Qwen/Qwen3-VL-8B-Thinking` (uses the CoT-trace discovery method, per this
  project's earlier finding that the standard method finds no causally-relevant
  heads for this checkpoint)
- `google/gemma-3n-E4B-it`
- `google/gemma-3n-E2B-it`

**Every parameter is a plain variable** -- edit and re-run freely. Pilot cell
(small N) runs first to catch bugs, then the main cell (full N).

In [1]:
%matplotlib inline
import gc
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from scipy import stats as sstats
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
from tqdm.auto import tqdm

from vis_head.common import DEFAULT_SEED
from vis_head.vir import aggregate_region_attention, collect_last_query_attentions, collect_cot_trace_region_attention, rank_heads_by_score
from vis_head.imagenet_grid import DEFAULT_IMAGENET_ROOT, list_val_class_dirs, load_class_names, sample_grid
from vis_head.modeling import decode_generated_text, find_image_token_range, load_model_and_processor, model_dims, prepare_inputs, run_generation
from vis_head.regions import assign_grid_cells_to_tokens, region_positions_from_ids
from vis_head.steering import group_heads_by_layer, intervention_positions, make_static_attention_mask_hook, register_mask_hooks, remove_handles

DEVICE = "cuda:0"
SEED = DEFAULT_SEED
ROWS, COLS = 2, 2
N_CELLS = ROWS * COLS
CELL_SIZE = 256
N_OPTIONS = 4
OPTION_LETTERS = ["A", "B", "C", "D"]
TOP_K = 15
DISCOVERY_PROMPT = lambda name: f"Identify the {name}."

CHECKPOINTS = {
    "qwen3_instruct": {"model_id": "Qwen/Qwen3-VL-8B-Instruct", "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
    "qwen3_thinking":  {"model_id": "Qwen/Qwen3-VL-8B-Thinking", "cot_discovery": True,  "max_new_tokens": 600, "cot_discovery_max_new_tokens": 150},
    "gemma3n_e4b_it":  {"model_id": "google/gemma-3n-E4B-it",    "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
    "gemma3n_e2b_it":  {"model_id": "google/gemma-3n-E2B-it",    "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
    "gemma3n_e4b_pt":  {"model_id": "google/gemma-3n-E4B",       "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
    "gemma3n_e2b_pt":  {"model_id": "google/gemma-3n-E2B",       "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
    "gemma4_e4b_it":   {"model_id": "google/gemma-4-E4B-it",     "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
    "gemma4_e4b_pt":   {"model_id": "google/gemma-4-E4B",        "cot_discovery": False, "max_new_tokens": 6, "cot_discovery_max_new_tokens": None},
}
N_DISCOVERY = 300
N_CAUSAL = 300
GEMMA_IMAGE_TOKEN_ID = 262145
GEMMA_GRID_SIDE = 16   # 256 soft image tokens = 16x16

imagenet_class_dirs = list_val_class_dirs(DEFAULT_IMAGENET_ROOT)
imagenet_class_names = load_class_names(DEFAULT_IMAGENET_ROOT)
print(f"{len(imagenet_class_dirs)} ImageNet classes available")


def get_letter_token_ids(tokenizer, letters):
    ids = {}
    for letter in letters:
        candidates = set()
        for cand in (letter, f" {letter}"):
            enc = tokenizer.encode(cand, add_special_tokens=False)
            if len(enc) == 1:
                candidates.add(enc[0])
        ids[letter] = sorted(candidates)
    return ids


def macro_f1(y_true, y_pred):
    labels = sorted(set(y_true) | set(y_pred))
    return f1_score(y_true, y_pred, labels=labels, average="macro")


def macro_auc(y_true, probs, classes):
    y_true_bin = label_binarize(y_true, classes=classes)
    try:
        return roc_auc_score(y_true_bin, probs, average="macro", multi_class="ovr")
    except ValueError:
        return float("nan")


def mcnemar_p(cond_a, cond_b):
    b = sum(1 for a, c in zip(cond_a, cond_b) if not a["correct"] and c["correct"])
    c = sum(1 for a, c in zip(cond_a, cond_b) if a["correct"] and not c["correct"])
    n = b + c
    if n == 0:
        return float("nan")
    stat = (abs(b - c) - 1) ** 2 / n
    return float(sstats.chi2.sf(stat, df=1))


def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()


def is_gemma(model_id):
    return "gemma" in model_id.lower()


def build_mcq_sample(rng):
    grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    target_cell = int(rng.randint(N_CELLS))
    correct_name = grid.cell_names[target_cell]
    other_names = [n for i, n in enumerate(grid.cell_names) if i != target_cell]
    distractor_idx = rng.choice(len(other_names), size=min(N_OPTIONS - 1, len(other_names)), replace=False)
    distractors = [other_names[i] for i in distractor_idx]
    options = [correct_name] + distractors
    order = rng.permutation(len(options))
    options = [options[i] for i in order]
    correct_letter = OPTION_LETTERS[int(np.where(order == 0)[0][0])]
    option_lines = "  ".join(f"{l}) {n}" for l, n in zip(OPTION_LETTERS[:len(options)], options))
    prompt = f"Which of the following is shown in this image? {option_lines}. Answer with only the letter."
    return {"grid": grid, "target_cell": target_cell, "options": options, "correct_letter": correct_letter, "prompt": prompt}


all_results = {}   # (model_tag, "PILOT"/"MAIN") -> list of per-condition summary rows

1000 ImageNet classes available


## Core runner: discovery (two selection criteria) + causal MCQ comparison

In [2]:
import vis_head.vir as _vir_module

# Two distinct Gemma multimodal architectures share the is_gemma() dispatch
# but use different image-token ids and template sources -- Gemma-3n (fixed
# 16x16=256 soft tokens, image_token_id=262145) and Gemma-4 (aspect-adaptive
# vision pooling that, for this project's fixed-size 2x2 composite grid
# images specifically, also empirically yields exactly 256 contiguous tokens
# in a genuine row-major 16x16 raster -- verified directly against
# Gemma4VisionPooler's kernel-index computation -- image_token_id=258880).
GEMMA_FAMILY_IMAGE_TOKEN_IDS = {"gemma3n": GEMMA_IMAGE_TOKEN_ID, "gemma4": 258880}
GEMMA_FAMILY_TEMPLATE_SOURCE = {"gemma3n": "gemma3n_e4b_it", "gemma4": "gemma4_e4b_it"}


def gemma_family(model_id):
    ml = model_id.lower()
    return "gemma4" if "gemma-4" in ml or "gemma4" in ml else "gemma3n"


def _gemma_find_image_token_range(inputs, processor=None):
    ids = inputs["input_ids"][0].tolist()
    for token_id in GEMMA_FAMILY_IMAGE_TOKEN_IDS.values():
        positions = [i for i, t in enumerate(ids) if t == token_id]
        if positions:
            return positions[0], positions[-1] + 1
    raise ValueError("No Gemma image tokens found (checked gemma3n and gemma4 token ids).")


_GEMMA_TEMPLATE_PROCESSORS = {}


def _get_gemma_template_processor(model_id):
    # Base (pt) Gemma checkpoints have no chat template of their own; use the
    # matching-family -it checkpoint's processor purely to render the
    # templated text, while tokenizing/processing images with the actual
    # checkpoint's own processor (same workaround pattern used for
    # Qwen2-VL-base and Gemma-3n pt in the multistage notebook).
    family = gemma_family(model_id)
    if family not in _GEMMA_TEMPLATE_PROCESSORS:
        from transformers import AutoProcessor as _AP
        source_tag = GEMMA_FAMILY_TEMPLATE_SOURCE[family]
        _GEMMA_TEMPLATE_PROCESSORS[family] = _AP.from_pretrained(CHECKPOINTS[source_tag]["model_id"])
    return _GEMMA_TEMPLATE_PROCESSORS[family]


def needs_gemma_template_workaround(model_id):
    return is_gemma(model_id) and not model_id.lower().endswith("-it")


def prepare_inputs_any(processor, image, prompt, model_id):
    if is_gemma(model_id):
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
        if needs_gemma_template_workaround(model_id):
            template_processor = _get_gemma_template_processor(model_id)
            text = template_processor.apply_chat_template([messages], tokenize=False, add_generation_prompt=True)
            text = text[0] if isinstance(text, list) else text
            return processor(text=text, images=[image], return_tensors="pt").to(DEVICE)
        return processor.apply_chat_template([messages], tokenize=True, add_generation_prompt=True,
                                              return_tensors="pt", return_dict=True).to(DEVICE)
    return prepare_inputs(processor, image, prompt, DEVICE)


def find_image_token_range_any(inputs, processor, model_id):
    if is_gemma(model_id):
        return _gemma_find_image_token_range(inputs, processor)
    return find_image_token_range(inputs, processor)


def assign_grid_any(rows, cols, inputs, model_id):
    if is_gemma(model_id):
        fake_thw = torch.tensor([[1, GEMMA_GRID_SIDE, GEMMA_GRID_SIDE]])
        return assign_grid_cells_to_tokens(image_grid_thw=fake_thw, rows=rows, cols=cols, spatial_merge=1)
    return assign_grid_cells_to_tokens(image_grid_thw=inputs["image_grid_thw"], rows=rows, cols=cols, spatial_merge=spatial_merge_global)


def run_selection_ablation(tag, cfg, n_discovery, n_causal):
    model_id = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id} ===")
    global spatial_merge_global
    if is_gemma(model_id):
        from transformers import AutoModelForImageTextToText, AutoProcessor as _AutoProc
        processor = _AutoProc.from_pretrained(model_id)
        model = AutoModelForImageTextToText.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, attn_implementation="eager"
        ).to(DEVICE)
        model.eval()
        n_layers = len(model.model.language_model.layers)
        n_heads = model.config.text_config.num_attention_heads
        spatial_merge_global = 1
        _vir_module.find_image_token_range = _gemma_find_image_token_range
    else:
        _vir_module.find_image_token_range = find_image_token_range
        model, processor = load_model_and_processor(model_id=model_id, device=DEVICE)
        n_layers, n_heads, spatial_merge_global = model_dims(model)
    print(f"{n_layers} layers x {n_heads} heads")

    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    # ---------------- discovery ----------------
    rng = np.random.RandomState(SEED)
    raw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for _ in tqdm(range(n_discovery), desc=f"[{tag}] discovery", leave=False):
        grid = sample_grid(rows=ROWS, cols=COLS, cell_size=CELL_SIZE, rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
        target_cell = int(rng.randint(N_CELLS))
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs_any(processor, grid.grid, prompt, model_id)
            region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
            if cfg["cot_discovery"]:
                img_start, img_end = find_image_token_range_any(inputs, processor, model_id)
                positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
                scores = collect_cot_trace_region_attention(
                    model=model, inputs=inputs, target_positions=positions[target_cell],
                    img_start=img_start, img_end=img_end, n_layers=n_layers, n_heads=n_heads,
                    max_new_tokens=cfg["cot_discovery_max_new_tokens"])
                raw_sum += scores
            else:
                attn = collect_last_query_attentions(model, inputs)
                region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
                raw_sum += region_attn[:, :, target_cell]
            valid += 1
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] discovery valid={valid}/{n_discovery}")
    vis_head_scores = (raw_sum / max(valid, 1)).astype(np.float32)

    # ---------------- two selection criteria ----------------
    ranked_raw = rank_heads_by_score(vis_head_scores)
    top_raw = [(r["layer"], r["head"]) for r in ranked_raw[:TOP_K]]

    per_layer_max = vis_head_scores.max(axis=1, keepdims=True)
    per_layer_max[per_layer_max == 0] = 1.0
    perlayer_norm_scores = vis_head_scores / per_layer_max
    ranked_perlayer = rank_heads_by_score(perlayer_norm_scores)
    top_perlayer = [(r["layer"], r["head"]) for r in ranked_perlayer[:TOP_K]]

    overlap = set(top_raw) & set(top_perlayer)
    print(f"  [{tag}] top-{TOP_K} overlap raw vs per-layer: {len(overlap)}/{TOP_K}")

    heads_raw = group_heads_by_layer(top_raw)
    heads_perlayer = group_heads_by_layer(top_perlayer)

    # ---------------- causal MCQ eval, shared samples ----------------
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]

    def run_mcq_generate(inputs, prompt_length, heads_by_layer, target_cell, max_new_tokens):
        handles = []
        if heads_by_layer is not None:
            img_start, img_end = find_image_token_range_any(inputs, processor, model_id)
            region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[target_cell]
            other_positions = [p for i in range(N_CELLS) if i != target_cell for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in heads_by_layer.items()
            }
            handles = register_mask_hooks(model, hook_by_layer)
        try:
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      output_scores=True, return_dict_in_generate=True)
        finally:
            remove_handles(handles)
        gen_ids = out.sequences[0, prompt_length:].tolist()
        answer_step = None
        for step in range(len(gen_ids) - 1, -1, -1):
            if gen_ids[step] in all_letter_ids_flat:
                answer_step = step
                break
        if answer_step is None:
            return None, np.zeros(N_OPTIONS)
        predicted = id_to_letter[gen_ids[answer_step]]
        step_logits = out.scores[answer_step][0].float()
        letter_logits = [step_logits[letter_token_ids[l]].max().item() for l in OPTION_LETTERS]
        probs = torch.softmax(torch.tensor(letter_logits), dim=0).numpy()
        return predicted, probs

    results = {"baseline": [], "raw": [], "perlayer": []}
    for sample in tqdm(mcq_samples, desc=f"[{tag}] MCQ", leave=False):
        inputs = prepare_inputs_any(processor, sample["grid"].grid, sample["prompt"], model_id)
        prompt_length = int(inputs["input_ids"].shape[1])
        for cond, heads in [("baseline", None), ("raw", heads_raw), ("perlayer", heads_perlayer)]:
            pred, probs = run_mcq_generate(inputs, prompt_length, heads, sample["target_cell"], cfg["max_new_tokens"])
            results[cond].append({"predicted": pred, "probs": probs, "correct": pred == sample["correct_letter"]})

    y_true = [s["correct_letter"] for s in mcq_samples]
    rows = []
    for cond in ["baseline", "raw", "perlayer"]:
        preds = [r["predicted"] if r["predicted"] else "UNPARSED" for r in results[cond]]
        probs = np.stack([r["probs"] for r in results[cond]])
        acc = accuracy_score(y_true, preds)
        f1 = macro_f1(y_true, preds)
        auc = macro_auc(y_true, probs, OPTION_LETTERS)
        rows.append({"model_tag": tag, "model_id": model_id, "condition": cond, "accuracy": acc, "f1": f1, "auc": auc,
                     "topk_overlap": len(overlap) if cond != "baseline" else None})
        print(f"  [{tag}] {cond:>10s}: acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}")

    p_raw_vs_base = mcnemar_p(results["baseline"], results["raw"])
    p_perlayer_vs_base = mcnemar_p(results["baseline"], results["perlayer"])
    p_raw_vs_perlayer = mcnemar_p(results["raw"], results["perlayer"])
    for row in rows:
        row["p_vs_baseline"] = {"baseline": float("nan"), "raw": p_raw_vs_base, "perlayer": p_perlayer_vs_base}[row["condition"]]
        row["p_raw_vs_perlayer"] = p_raw_vs_perlayer
    print(f"  [{tag}] p(raw vs base)={p_raw_vs_base:.3e}  p(perlayer vs base)={p_perlayer_vs_base:.3e}  p(raw vs perlayer)={p_raw_vs_perlayer:.3e}")

    free_gpu(model, processor)
    return rows

## Pilot run (small N, catches bugs fast)

In [3]:
PILOT_N_DISCOVERY = 20
PILOT_N_CAUSAL = 20

for tag, cfg in CHECKPOINTS.items():
    rows = run_selection_ablation(tag, cfg, PILOT_N_DISCOVERY, PILOT_N_CAUSAL)
    all_results[(tag, "PILOT")] = rows

pd.DataFrame([r for k, rows in all_results.items() if k[1] == "PILOT" for r in rows])


=== [qwen3_instruct] Loading Qwen/Qwen3-VL-8B-Instruct ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

36 layers x 32 heads


...

36 layers x 32 heads


[qwen3_thinking] discovery:   0%|          | 0/20 [00:00<?, ?it/s]

  [qwen3_thinking] discovery valid=20/20
  [qwen3_thinking] top-15 overlap raw vs per-layer: 1/15


[qwen3_thinking] MCQ:   0%|          | 0/20 [00:00<?, ?it/s]

: 

## Main run (N_DISCOVERY / N_CAUSAL, default 300/300)

In [3]:
import os

csv_path = "logs/raw_vs_perlayer_selection_ablation.csv"
os.makedirs("logs", exist_ok=True)
already_done = set()
if os.path.exists(csv_path):
    _prev = pd.read_csv(csv_path)
    already_done = set(_prev["model_tag"].unique())
    print(f"Resuming: already have {sorted(already_done)}")

for tag, cfg in CHECKPOINTS.items():
    if tag in already_done:
        print(f"Skipping {tag}, already in {csv_path}")
        continue
    rows = run_selection_ablation(tag, cfg, N_DISCOVERY, N_CAUSAL)
    all_results[(tag, "MAIN")] = rows
    df_rows = pd.DataFrame(rows)
    header = not os.path.exists(csv_path)
    df_rows.to_csv(csv_path, mode="a", header=header, index=False)
    print(f"  [{tag}] appended to {csv_path}")

final_df = pd.read_csv(csv_path)
pd.set_option("display.width", 200)
print(final_df.to_string(index=False))
print(f"\nSaved to {csv_path}")

Resuming: already have ['gemma3n_e2b_it', 'gemma3n_e4b_it', 'qwen3_instruct', 'qwen3_thinking']
Skipping qwen3_instruct, already in logs/raw_vs_perlayer_selection_ablation.csv
Skipping qwen3_thinking, already in logs/raw_vs_perlayer_selection_ablation.csv
Skipping gemma3n_e4b_it, already in logs/raw_vs_perlayer_selection_ablation.csv
Skipping gemma3n_e2b_it, already in logs/raw_vs_perlayer_selection_ablation.csv

=== [gemma3n_e4b_pt] Loading google/gemma-3n-E4B ===


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1631 [00:00<?, ?it/s]

35 layers x 8 heads


[gemma3n_e4b_pt] discovery:   0%|          | 0/300 [00:00<?, ?it/s]

...

[gemma4_e4b_pt] discovery:   0%|          | 0/300 [00:00<?, ?it/s]

  [gemma4_e4b_pt] discovery valid=300/300
  [gemma4_e4b_pt] top-15 overlap raw vs per-layer: 3/15


[gemma4_e4b_pt] MCQ:   0%|          | 0/300 [00:00<?, ?it/s]

  [gemma4_e4b_pt]   baseline: acc=0.000  f1=0.000  auc=0.499
  [gemma4_e4b_pt]        raw: acc=0.000  f1=0.000  auc=0.500
  [gemma4_e4b_pt]   perlayer: acc=0.000  f1=0.000  auc=0.500
  [gemma4_e4b_pt] p(raw vs base)=nan  p(perlayer vs base)=nan  p(raw vs perlayer)=nan


  [gemma4_e4b_pt] appended to logs/raw_vs_perlayer_selection_ablation.csv
     model_tag                  model_id condition  accuracy       f1      auc  topk_overlap  p_vs_baseline  p_raw_vs_perlayer
qwen3_instruct Qwen/Qwen3-VL-8B-Instruct  baseline  0.246667 0.245089 0.487004           NaN            NaN       2.895571e-29
qwen3_instruct Qwen/Qwen3-VL-8B-Instruct       raw  0.706667 0.703391 0.867193           4.0   3.452221e-30       2.895571e-29
qwen3_instruct Qwen/Qwen3-VL-8B-Instruct  perlayer  0.243333 0.239478 0.492713           4.0   1.000000e+00       2.895571e-29
qwen3_thinking Qwen/Qwen3-VL-8B-Thinking  baseline  0.233333 0.229273 0.498794           NaN            NaN       3.653846e-26
qwen3_thinking Qwen/Qwen3-VL-8B-Thinking       raw  0.656667 0.657984 0.735282           1.0   4.478296e-24       3.653846e-26
qwen3_thinking Qwen/Qwen3-VL-8B-Thinking  perlayer  0.210000 0.206663 0.475311           1.0   4.996423e-01       3.653846e-26
gemma3n_e4b_it    google/gemma-3n-E4B

## Part 2 -- `imagenet_circular_grid`: a position-controlled discovery dataset

`sample_grid` (used above) varies BOTH which object appears AND which cell it
lands in, independently, every sample -- good for generality but confounds
"this head fires on this object" with "this head fires at this position."

`imagenet_circular_grid` (new, `vis_head.imagenet_grid.build_imagenet_circular_grid_dataset`)
isolates position as the only variable: for each of `p` target objects, the
SAME target image and the SAME set of distractor images are tiled into all
`n*m` grid positions in turn (`p*n*m` grids total). A head's raw discovery
score should look the same across all `n*m` variants of the same object if it
is really doing position-conditioned routing; if the score swings wildly
across positions for identical content, that swing is noise, not signal --
which is exactly what the new `snr_score` (mean/std across samples) is built
to detect, and this dataset is the cleanest place to compute it, since content
is held fixed and only position varies.

Selection methods compared here (5, extending the raw-vs-perlayer comparison
above): **raw**, **perlayer_max**, **perlayer_zscore**, **snr** (mean/std of
the per-object-per-position raw score, new this section), and
**value_weighted** (Kobayashi et al. attention x value-norm).

Causal steering from each of the 5 resulting head selections is then measured
on the ORIGINAL `sample_grid` MCQ task (not the circular-grid images
themselves) -- i.e. "does a head selection derived from position-controlled
discovery transfer to the same free-form causal eval used everywhere else in
this project?" -- via both grid MCQ generation accuracy and linear-probe
classification on final-layer hidden states.

In [ ]:
from vis_head.imagenet_grid import build_imagenet_circular_grid_dataset
from vis_head.vir import collect_last_query_attentions_value_weighted, score_variance_stats, snr_score, normalize_scores_per_layer_max, normalize_scores_per_layer_zscore
import vis_head.modeling as _modeling_module
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def load_any_model(model_id):
    global spatial_merge_global
    if is_gemma(model_id):
        from transformers import AutoModelForImageTextToText, AutoProcessor as _AutoProc
        processor = _AutoProc.from_pretrained(model_id)
        model = AutoModelForImageTextToText.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, attn_implementation="eager"
        ).to(DEVICE)
        model.eval()
        n_layers = len(model.model.language_model.layers)
        n_heads = model.config.text_config.num_attention_heads
        spatial_merge_global = 1
        _vir_module.find_image_token_range = _gemma_find_image_token_range
        _modeling_module.find_image_token_range = _gemma_find_image_token_range
    else:
        model, processor = load_model_and_processor(model_id=model_id, device=DEVICE)
        n_layers, n_heads, spatial_merge_global = model_dims(model)
        _vir_module.find_image_token_range = find_image_token_range
        _modeling_module.find_image_token_range = find_image_token_range
    return model, processor, n_layers, n_heads


def get_final_hidden_state(model, inputs):
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, use_cache=False)
    last_layer = out.hidden_states[-1]
    if last_layer.dim() == 4:
        # Gemma-3n AltUp: (num_altup_inputs, batch, seq, hidden) -- index 0 is
        # config.altup_active_idx, the stream actually used for output logits.
        return last_layer[0, 0, -1, :].float().cpu().numpy()
    return last_layer[0, -1, :].float().cpu().numpy()


def fit_and_eval_probe(X, y, seed=SEED):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = LogisticRegression(max_iter=2000, multi_class="multinomial", C=1.0)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    y_proba = clf.predict_proba(X_test)
    y_test_bin = label_binarize(y_test, classes=clf.classes_)
    try:
        auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")
    return {"accuracy": acc, "auc": auc, "n_train": len(y_train), "n_test": len(y_test)}


def run_mcq_generate_g(model, processor, letter_token_ids, all_letter_ids_flat, id_to_letter, n_heads, model_id,
                        inputs, prompt_length, heads_by_layer, target_cell, max_new_tokens=6):
    handles = []
    if heads_by_layer is not None:
        img_start, img_end = find_image_token_range_any(inputs, processor, model_id)
        region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
        positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = positions[target_cell]
        other_positions = [p for i in range(N_CELLS) if i != target_cell for p in positions[i]]
        suppress_positions, boost_positions, pad = intervention_positions(
            mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
            img_start=img_start, img_end=img_end, prompt_length=prompt_length)
        hook_by_layer = {
            l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                boost_positions=boost_positions, n_query_heads=n_heads,
                                                device=DEVICE, decode_only=False, pad_with_suppress=pad)
            for l, hh in heads_by_layer.items()
        }
        handles = register_mask_hooks(model, hook_by_layer)
    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  output_scores=True, return_dict_in_generate=True)
    finally:
        remove_handles(handles)
    gen_ids = out.sequences[0, prompt_length:].tolist()
    answer_step = None
    for step in range(len(gen_ids) - 1, -1, -1):
        if gen_ids[step] in all_letter_ids_flat:
            answer_step = step
            break
    if answer_step is None:
        return None, np.zeros(N_OPTIONS)
    predicted = id_to_letter[gen_ids[answer_step]]
    step_logits = out.scores[answer_step][0].float()
    letter_logits = [step_logits[letter_token_ids[l]].max().item() for l in OPTION_LETTERS]
    probs = torch.softmax(torch.tensor(letter_logits), dim=0).numpy()
    return predicted, probs

def mcnemar_p_bool(cond_a, cond_b):
    b = sum(1 for a, c in zip(cond_a, cond_b) if not a and c)
    c = sum(1 for a, c in zip(cond_a, cond_b) if a and not c)
    n = b + c
    if n == 0:
        return float("nan")
    stat = (abs(b - c) - 1) ** 2 / n
    return float(sstats.chi2.sf(stat, df=1))


# Restricted to the Gemma-3n pair this diagnostic work is actually about --
# Part 1 (raw vs per-layer) already ran all 4 checkpoints; Part 2/3 focus on
# just these two so results land faster and can be checked incrementally.
PART23_CHECKPOINTS = {
    "gemma4_e4b_pt": CHECKPOINTS["gemma4_e4b_pt"],
    "gemma4_e4b_it": CHECKPOINTS["gemma4_e4b_it"],
}


In [ ]:
def run_circular_grid_experiment(tag, cfg, n_classes, n_causal):
    model_id = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id} for imagenet_circular_grid discovery ===")
    model, processor, n_layers, n_heads = load_any_model(model_id)
    print(f"{n_layers} layers x {n_heads} heads")

    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    # ---------------- circular-grid discovery ----------------
    rng = np.random.RandomState(SEED + 900)
    circular_dataset = build_imagenet_circular_grid_dataset(
        n_classes=n_classes, rows=ROWS, cols=COLS, cell_size=CELL_SIZE,
        rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    print(f"  [{tag}] imagenet_circular_grid: {n_classes} objects x {N_CELLS} positions = {len(circular_dataset)} grids")

    per_sample_raw = []
    vw_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid = 0
    for grid, target_cell in tqdm(circular_dataset, desc=f"[{tag}] circular discovery", leave=False):
        prompt = DISCOVERY_PROMPT(grid.cell_names[target_cell])
        try:
            inputs = prepare_inputs_any(processor, grid.grid, prompt, model_id)
            region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
            attn = collect_last_query_attentions(model, inputs)
            region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            per_sample_raw.append(region_attn[:, :, target_cell])

            vw_attn = collect_last_query_attentions_value_weighted(model, inputs, processor)
            vw_region_attn = aggregate_region_attention(attn_at_query=vw_attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
            vw_sum += vw_region_attn[:, :, target_cell]
            valid += 1
        except Exception as exc:
            print(f"  Skipping grid: {exc}")
    print(f"  [{tag}] circular discovery valid={valid}/{len(circular_dataset)}")

    per_sample_raw = np.stack(per_sample_raw).astype(np.float32)   # (n_samples, n_layers, n_heads)
    raw_scores = per_sample_raw.mean(axis=0)
    vw_scores = (vw_sum / max(valid, 1)).astype(np.float32)
    var_stats = score_variance_stats(per_sample_raw)
    snr_scores = snr_score(per_sample_raw)

    method_scores = {
        "raw": raw_scores,
        "perlayer_max": normalize_scores_per_layer_max(raw_scores),
        "perlayer_zscore": normalize_scores_per_layer_zscore(raw_scores),
        "snr": snr_scores,
        "value_weighted": vw_scores,
    }
    selections = {}
    for name, mat in method_scores.items():
        ranked = rank_heads_by_score(mat)
        top = [(r["layer"], r["head"]) for r in ranked[:TOP_K]]
        selections[name] = group_heads_by_layer(top)

    print(f"  [{tag}] avg per-head CV (std/mean) across circular positions: {np.nanmean(var_stats['cv'][np.isfinite(var_stats['cv'])]):.3f}")

    # ---------------- causal eval on the ORIGINAL sample_grid MCQ task ----------------
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]
    y_true = [s["correct_letter"] for s in mcq_samples]

    baseline_correct, baseline_hidden = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] baseline", leave=False):
        inputs = prepare_inputs_any(processor, sample["grid"].grid, sample["prompt"], model_id)
        prompt_length = int(inputs["input_ids"].shape[1])
        pred, _ = run_mcq_generate_g(model, processor, letter_token_ids, all_letter_ids_flat, id_to_letter, n_heads, model_id,
                                      inputs, prompt_length, None, sample["target_cell"], cfg["max_new_tokens"])
        baseline_correct.append(pred == sample["correct_letter"])
        baseline_hidden.append(get_final_hidden_state(model, inputs))
    baseline_hidden = np.stack(baseline_hidden)
    probe_baseline = fit_and_eval_probe(baseline_hidden, y_true)
    mcq_acc_baseline = float(np.mean(baseline_correct))
    print(f"  [{tag}] baseline: mcq_acc={mcq_acc_baseline:.3f}  probe_acc={probe_baseline['accuracy']:.3f}")

    rows = []
    for method_name, heads_by_layer in selections.items():
        steer_correct, steer_hidden = [], []
        for sample in tqdm(mcq_samples, desc=f"[{tag}] {method_name}", leave=False):
            inputs = prepare_inputs_any(processor, sample["grid"].grid, sample["prompt"], model_id)
            prompt_length = int(inputs["input_ids"].shape[1])
            pred, _ = run_mcq_generate_g(model, processor, letter_token_ids, all_letter_ids_flat, id_to_letter, n_heads, model_id,
                                          inputs, prompt_length, heads_by_layer, sample["target_cell"], cfg["max_new_tokens"])
            steer_correct.append(pred == sample["correct_letter"])

            img_start, img_end = find_image_token_range_any(inputs, processor, model_id)
            region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
            positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
            target_positions = positions[sample["target_cell"]]
            other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
            suppress_positions, boost_positions, pad = intervention_positions(
                mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
                img_start=img_start, img_end=img_end, prompt_length=prompt_length)
            hook_by_layer = {
                l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                    boost_positions=boost_positions, n_query_heads=n_heads,
                                                    device=DEVICE, decode_only=False, pad_with_suppress=pad)
                for l, hh in heads_by_layer.items()
            }
            handles = register_mask_hooks(model, hook_by_layer)
            try:
                steer_hidden.append(get_final_hidden_state(model, inputs))
            finally:
                remove_handles(handles)

        steer_hidden = np.stack(steer_hidden)
        probe_steer = fit_and_eval_probe(steer_hidden, y_true)
        mcq_acc_steer = float(np.mean(steer_correct))
        p_mcnemar = mcnemar_p_bool(baseline_correct, steer_correct)

        rows.append({
            "model_tag": tag, "model_id": model_id, "method": method_name,
            "mcq_acc_baseline": mcq_acc_baseline, "mcq_acc_steered": mcq_acc_steer,
            "mcq_delta": mcq_acc_steer - mcq_acc_baseline, "mcq_p_mcnemar": p_mcnemar,
            "probe_acc_baseline": probe_baseline["accuracy"], "probe_acc_steered": probe_steer["accuracy"],
            "probe_delta": probe_steer["accuracy"] - probe_baseline["accuracy"],
            "avg_head_cv": float(np.nanmean(var_stats["cv"][np.isfinite(var_stats["cv"])])),
        })
        print(f"  [{tag}] {method_name:>16s}: mcq {mcq_acc_baseline:.3f}->{mcq_acc_steer:.3f} (p={p_mcnemar:.2e})   "
              f"probe {probe_baseline['accuracy']:.3f}->{probe_steer['accuracy']:.3f}")

    free_gpu(model, processor)
    return rows

### Pilot (small n_classes, catches bugs fast)

In [ ]:
PILOT_CIRCULAR_N_CLASSES = 5    # 5 objects x 4 positions = 20 discovery grids
PILOT_CIRCULAR_N_CAUSAL = 20

circular_pilot_rows = []
for tag, cfg in PART23_CHECKPOINTS.items():
    rows = run_circular_grid_experiment(tag, cfg, PILOT_CIRCULAR_N_CLASSES, PILOT_CIRCULAR_N_CAUSAL)
    circular_pilot_rows.extend(rows)
    print(pd.DataFrame(rows).to_string(index=False), flush=True)

circular_pilot_df = pd.DataFrame(circular_pilot_rows)
circular_pilot_df

### Main run (n_classes=75 -> 300 discovery grids per checkpoint, matching N_CAUSAL=300 above)

In [ ]:
MAIN_CIRCULAR_N_CLASSES = 75    # 75 objects x 4 positions = 300 discovery grids
MAIN_CIRCULAR_N_CAUSAL = 300

import os
circular_csv = "logs/imagenet_circular_grid_head_selection.csv"
os.makedirs("logs", exist_ok=True)
_already = set()
if os.path.exists(circular_csv):
    _already = set(pd.read_csv(circular_csv)["model_tag"].unique())
    print(f"Resuming: already have {sorted(_already)}", flush=True)

circular_main_rows = []
for tag, cfg in PART23_CHECKPOINTS.items():
    if tag in _already:
        print(f"Skipping {tag}, already in {circular_csv}", flush=True)
        continue
    rows = run_circular_grid_experiment(tag, cfg, MAIN_CIRCULAR_N_CLASSES, MAIN_CIRCULAR_N_CAUSAL)
    circular_main_rows.extend(rows)
    df_rows = pd.DataFrame(rows)
    header = not os.path.exists(circular_csv)
    df_rows.to_csv(circular_csv, mode="a", header=header, index=False)
    print(f"[{tag}] appended to {circular_csv}:", flush=True)
    print(df_rows.to_string(index=False), flush=True)

circular_main_df = pd.read_csv(circular_csv)
pd.set_option("display.width", 200)
print(circular_main_df.to_string(index=False))
print(f"\nSaved to {circular_csv}")

### Interpretation

Compare `mcq_delta`/`probe_delta` per (checkpoint, method) here against the
`sample_grid`-discovered raw-vs-perlayer results in Part 1 above. If a method
that performed poorly on random-grid discovery (e.g. raw for Gemma-3n) does
noticeably better when discovered from the position-controlled
`imagenet_circular_grid` dataset instead, that points at discovery-data
composition (not just the scoring formula) as part of the earlier Gemma-3n
puzzle. The `avg_head_cv` column reports how noisy each checkpoint's
raw per-head scores are across positions of the SAME object -- a high value
means many heads' apparent selectivity is unstable even when content is held
fixed, i.e. genuinely low signal-to-noise, not just hard to find.

## Part 3 -- Visual Head Score algorithm (spatial sensitivity + localization + correspondence + instruction selectivity, permutation-tested, FDR-corrected)

Implements the requested pipeline end to end, reusing the `imagenet_circular_grid`
dataset built in Part 2 (object moved to every grid position) and the 3
best-performing discovery-prompt phrasings identified earlier in the project's
18-phrasing sweep:

```
FOR each image: move object to every grid position
  FOR each prompt: run the VLM
    FOR each layer/head: extract attention to visual tokens, compute
      1. object attention  2. control-corrected attention
      3. spatial variance  4. object localization  5. spatial correspondence
AFTER all images: compute instruction selectivity per head
  normalize all components, Visual Head Score = sum of z-scored components
  permutation test -> FDR correction -> select significant high-scoring heads
```

All 5 per-sample components and the aggregate scores are computed by the new
`vis_head.vir.compute_visual_head_scores` (steps 1-5 + instruction
selectivity + normalized composite) and
`vis_head.vir.permutation_test_visual_head_scores` (permutation null +
Benjamini-Hochberg FDR, both new library functions this session). The GPU loop
below only collects the raw per-sample, per-cell attention tensor
`(n_samples, n_layers, n_heads, n_cells)` -- everything downstream is pure
numpy on that cached tensor, so the permutation test (200 shuffles) costs no
extra model forward passes.

In [ ]:
from vis_head.vir import permutation_test_visual_head_scores

VHS_PROMPTS = {
    "what_shows": lambda name: f"What shows the {name}?",
    "find":       lambda name: f"Find the {name}.",
    "identify":   lambda name: f"Identify the {name}.",
}
VHS_N_PERMUTATIONS = 200
VHS_ALPHA = 0.05
# Debugging finding (see raw_vs_perlayer notebook discussion): heads with the
# highest observational spatial_correspondence/object_localization are
# concentrated in LATE layers (close to the output) for Gemma-3n -- boosting
# them directly corrupts the near-final computation instead of giving
# downstream layers a chance to integrate the injected signal, which collapsed
# causal MCQ accuracy to ~0. The empirically steerable heads (found by
# per-layer-max normalization in Part 1/2) are concentrated in early/mid
# layers instead, with near-chance observational correspondence scores.
# "Correlates with object position" and "is steerable when boosted" are
# different properties -- restrict the VHS candidate pool to the first
# VHS_MAX_LAYER_FRAC of layers so the composite score can't select heads from
# the collapse-prone late-layer zone.
VHS_MAX_LAYER_FRAC = 0.6


def collect_visual_head_score_tensor(model, processor, model_id, dataset, prompts):
    """Run every (image-with-object-at-position, prompt) pair once, returning
    the full per-sample, per-cell region-attention tensor plus target-cell and
    prompt-id arrays needed by `compute_visual_head_scores`."""
    region_attn_samples, target_cells, prompt_ids = [], [], []
    prompt_names = list(prompts.keys())
    for grid, target_cell in tqdm(dataset, desc="VHS discovery (images)", leave=False):
        for p_idx, prompt_name in enumerate(prompt_names):
            prompt = prompts[prompt_name](grid.cell_names[target_cell])
            try:
                inputs = prepare_inputs_any(processor, grid.grid, prompt, model_id)
                region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
                attn = collect_last_query_attentions(model, inputs)
                region_attn = aggregate_region_attention(attn_at_query=attn, inputs=inputs, processor=processor, region_ids=region_ids, n_regions=N_CELLS)
                # region_attn: (n_layers, n_heads, n_cells) -- full per-cell attention, not just target
                per_cell = np.stack([region_attn[:, :, c] for c in range(N_CELLS)], axis=-1)
                region_attn_samples.append(per_cell)
                target_cells.append(target_cell)
                prompt_ids.append(p_idx)
            except Exception as exc:
                print(f"  Skipping sample: {exc}")
    region_attn_all = np.stack(region_attn_samples).astype(np.float32)   # (n_samples, n_layers, n_heads, n_cells)
    return region_attn_all, np.array(target_cells), np.array(prompt_ids), prompt_names


def run_visual_head_score_experiment(tag, cfg, n_classes, n_causal):
    model_id = cfg["model_id"]
    print(f"\n=== [{tag}] Loading {model_id} for Visual Head Score algorithm ===")
    model, processor, n_layers, n_heads = load_any_model(model_id)
    print(f"{n_layers} layers x {n_heads} heads")

    letter_token_ids = get_letter_token_ids(processor.tokenizer, OPTION_LETTERS)
    all_letter_ids_flat = set(i for ids in letter_token_ids.values() for i in ids)
    id_to_letter = {i: l for l, ids in letter_token_ids.items() for i in ids}

    rng = np.random.RandomState(SEED + 1300)
    dataset = build_imagenet_circular_grid_dataset(
        n_classes=n_classes, rows=ROWS, cols=COLS, cell_size=CELL_SIZE,
        rng=rng, class_dirs=imagenet_class_dirs, class_names=imagenet_class_names)
    print(f"  [{tag}] {n_classes} objects x {N_CELLS} positions x {len(VHS_PROMPTS)} prompts "
          f"= {len(dataset) * len(VHS_PROMPTS)} (image, prompt) samples")

    region_attn_all, target_cells, prompt_ids, prompt_names = collect_visual_head_score_tensor(
        model, processor, model_id, dataset, VHS_PROMPTS)
    print(f"  [{tag}] collected tensor {region_attn_all.shape}")

    perm_rng = np.random.RandomState(SEED + 1400)
    vhs = permutation_test_visual_head_scores(
        region_attn_all, target_cells, prompt_ids,
        n_permutations=VHS_N_PERMUTATIONS, rng=perm_rng, alpha=VHS_ALPHA)

    n_significant = int(vhs["significant_mask"].sum())
    print(f"  [{tag}] {n_significant}/{n_layers * n_heads} heads significant at FDR q<={VHS_ALPHA}")

    max_layer = int(n_layers * VHS_MAX_LAYER_FRAC)
    print(f"  [{tag}] restricting candidates to layers < {max_layer} (of {n_layers}) to avoid late-layer collapse")

    ranked = rank_heads_by_score(vhs["visual_head_score"])
    ranked = [r for r in ranked if r["layer"] < max_layer]
    significant_ranked = [r for r in ranked if vhs["significant_mask"][r["layer"], r["head"]]]
    if len(significant_ranked) >= 1:
        selected = significant_ranked[:max(TOP_K, len(significant_ranked))] if len(significant_ranked) < TOP_K else significant_ranked[:TOP_K]
    else:
        print(f"  [{tag}] WARNING: no heads survived FDR correction, falling back to top-{TOP_K} by raw visual_head_score")
        selected = ranked[:TOP_K]
    top_vhs = [(r["layer"], r["head"]) for r in selected]
    heads_vhs = group_heads_by_layer(top_vhs)
    print(f"  [{tag}] selected {len(top_vhs)} heads: {top_vhs}")

    # ---------------- causal eval on the ORIGINAL sample_grid MCQ task ----------------
    mcq_rng = np.random.RandomState(SEED + 555)
    mcq_samples = [build_mcq_sample(mcq_rng) for _ in range(n_causal)]
    y_true = [s["correct_letter"] for s in mcq_samples]

    baseline_correct, baseline_hidden = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] VHS baseline", leave=False):
        inputs = prepare_inputs_any(processor, sample["grid"].grid, sample["prompt"], model_id)
        prompt_length = int(inputs["input_ids"].shape[1])
        pred, _ = run_mcq_generate_g(model, processor, letter_token_ids, all_letter_ids_flat, id_to_letter, n_heads, model_id,
                                      inputs, prompt_length, None, sample["target_cell"], cfg["max_new_tokens"])
        baseline_correct.append(pred == sample["correct_letter"])
        baseline_hidden.append(get_final_hidden_state(model, inputs))
    baseline_hidden = np.stack(baseline_hidden)
    probe_baseline = fit_and_eval_probe(baseline_hidden, y_true)
    mcq_acc_baseline = float(np.mean(baseline_correct))

    steer_correct, steer_hidden = [], []
    for sample in tqdm(mcq_samples, desc=f"[{tag}] VHS steered", leave=False):
        inputs = prepare_inputs_any(processor, sample["grid"].grid, sample["prompt"], model_id)
        prompt_length = int(inputs["input_ids"].shape[1])
        pred, _ = run_mcq_generate_g(model, processor, letter_token_ids, all_letter_ids_flat, id_to_letter, n_heads, model_id,
                                      inputs, prompt_length, heads_vhs, sample["target_cell"], cfg["max_new_tokens"])
        steer_correct.append(pred == sample["correct_letter"])

        img_start, img_end = find_image_token_range_any(inputs, processor, model_id)
        region_ids, _ = assign_grid_any(ROWS, COLS, inputs, model_id)
        positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=N_CELLS)
        target_positions = positions[sample["target_cell"]]
        other_positions = [p for i in range(N_CELLS) if i != sample["target_cell"] for p in positions[i]]
        suppress_positions, boost_positions, pad = intervention_positions(
            mode="boost_suppress", target_positions=target_positions, other_image_positions=other_positions,
            img_start=img_start, img_end=img_end, prompt_length=prompt_length)
        hook_by_layer = {
            l: make_static_attention_mask_hook(head_indices=hh, suppress_positions=suppress_positions,
                                                boost_positions=boost_positions, n_query_heads=n_heads,
                                                device=DEVICE, decode_only=False, pad_with_suppress=pad)
            for l, hh in heads_vhs.items()
        }
        handles = register_mask_hooks(model, hook_by_layer)
        try:
            steer_hidden.append(get_final_hidden_state(model, inputs))
        finally:
            remove_handles(handles)

    steer_hidden = np.stack(steer_hidden)
    probe_steer = fit_and_eval_probe(steer_hidden, y_true)
    mcq_acc_steer = float(np.mean(steer_correct))
    p_mcnemar = mcnemar_p_bool(baseline_correct, steer_correct)

    print(f"  [{tag}] VHS-selected heads: mcq {mcq_acc_baseline:.3f}->{mcq_acc_steer:.3f} (p={p_mcnemar:.2e})   "
          f"probe {probe_baseline['accuracy']:.3f}->{probe_steer['accuracy']:.3f}")

    free_gpu(model, processor)
    return {
        "model_tag": tag, "model_id": model_id, "n_heads_significant": n_significant,
        "n_heads_selected": len(top_vhs), "top_heads": top_vhs,
        "mcq_acc_baseline": mcq_acc_baseline, "mcq_acc_steered": mcq_acc_steer,
        "mcq_delta": mcq_acc_steer - mcq_acc_baseline, "mcq_p_mcnemar": p_mcnemar,
        "probe_acc_baseline": probe_baseline["accuracy"], "probe_acc_steered": probe_steer["accuracy"],
        "probe_delta": probe_steer["accuracy"] - probe_baseline["accuracy"],
    }

### Pilot (small n_classes, catches bugs fast)

In [ ]:
PILOT_VHS_N_CLASSES = 5
PILOT_VHS_N_CAUSAL = 20

vhs_pilot_rows = []
for tag, cfg in PART23_CHECKPOINTS.items():
    row = run_visual_head_score_experiment(tag, cfg, PILOT_VHS_N_CLASSES, PILOT_VHS_N_CAUSAL)
    vhs_pilot_rows.append(row)
    print(pd.DataFrame([row]).drop(columns=["top_heads"]).to_string(index=False), flush=True)

vhs_pilot_df = pd.DataFrame(vhs_pilot_rows)
vhs_pilot_df

### Main run (n_classes=75 -> 300 images x 3 prompts = 900 discovery samples per checkpoint)

In [ ]:
MAIN_VHS_N_CLASSES = 75
MAIN_VHS_N_CAUSAL = 300

import os
vhs_csv = "logs/visual_head_score_algorithm.csv"
os.makedirs("logs", exist_ok=True)
_already = set()
if os.path.exists(vhs_csv):
    _already = set(pd.read_csv(vhs_csv)["model_tag"].unique())
    print(f"Resuming: already have {sorted(_already)}", flush=True)

vhs_main_rows = []
for tag, cfg in PART23_CHECKPOINTS.items():
    if tag in _already:
        print(f"Skipping {tag}, already in {vhs_csv}", flush=True)
        continue
    row = run_visual_head_score_experiment(tag, cfg, MAIN_VHS_N_CLASSES, MAIN_VHS_N_CAUSAL)
    vhs_main_rows.append(row)
    df_row = pd.DataFrame([row])
    header = not os.path.exists(vhs_csv)
    df_row.to_csv(vhs_csv, mode="a", header=header, index=False)
    print(f"[{tag}] appended to {vhs_csv}:", flush=True)
    print(df_row.drop(columns=["top_heads"]).to_string(index=False), flush=True)

vhs_main_df = pd.read_csv(vhs_csv)
pd.set_option("display.width", 200)
print(vhs_main_df.drop(columns=["top_heads"], errors="ignore").to_string(index=False))
print(f"\nSaved to {vhs_csv}")

### Interpretation

Compare `vhs_main_df`'s `mcq_delta`/`probe_delta` against Part 1 (raw vs.
per-layer, `sample_grid` discovery) and Part 2 (5-method sweep on
`imagenet_circular_grid`). The permutation-tested, FDR-corrected selection is
the most statistically conservative of the three approaches -- if it also
beats the plain top-K methods on Gemma-3n specifically, that is fairly strong
evidence the earlier weak/negative Gemma-3n-E4B-it steering result was a
head-selection-method problem rather than a model-capability problem;
if it performs no better, that shifts weight toward the model itself lacking
heads whose attention pattern is causally load-bearing for this task, no
matter how rigorously they're identified.

## Result

(filled in after running)